In [1]:
%load_ext autoreload
%autoreload 2

# --- Standard libs ---
import numpy as np
import pandas as pd
import time
import sys
import serial
from pathlib import Path

#
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# --- Project imports ---
from Instruments.GX_271.gilson_ethernet import GilsonEthernet
from Instruments.GX_271.tray import Tray
from Instruments.GX_271.rack import Rack_209, Rack_3dp
from Core.logging import flow_logger as logger
from Instruments.GX_271.dim import DIM
from Instruments.WPI_Syr_pump.Pump import AL1000
from Instruments.Runze_valves.Runze62Valve import Runze62Valve
from Ecosystems.reaction_segment_generation import RSG

In [3]:
# --- Instantiate serial object for AL1000 pump ----
ser = serial.Serial("COM2", 9600, timeout=0.5)
pump = AL1000(ser, device_address="@00", sleep_time=0.5)

pump.connect()

Device is connected


In [4]:
#--- Construct the tray ---
tray = Tray()

# --- Connect to Gilson ---
g = GilsonEthernet("192.168.1.101", admin_port=50185, tray=tray)

# --- Start logging (declare this run belongs to the experiment "Development") ---
logger.start_experiment("Development")

# --- Instantiate modules (racks, wash stations, etc.) (these know internal geometry) ---
rack1 = Rack_209()  
rack2 = Rack_3dp()
#DIM = DIM()

# --- Register modules on the tray with global offsets, previously handled by each module in the tray ---
tray.add_module(
    slot=1,
    name="rack1",
    module=rack1,
    x_offset=36,
    y_offset=8,
    x_min=27,
    x_max=127,
    y_min=2,
    y_max=292
)

tray.add_module(
    slot=2,
    name="rack2",
    module=rack2,
    x_offset=201,
    y_offset=39,
    x_min=157,
    x_max=247,
    y_min=2,
    y_max=292
) 

# Instantiate the RSG ecosystem with the connected Gilson and AL1000
rsg = RSG(gilson=g, pump=pump, syringe_diameter=4.606)

In [ ]:
import time

def run_prepickup_validation(
    rsg,
    analyte_volumes,
    start_vial_pos=17,
    n_replicates=3,
    prepickup_volume=20,
):
    """
    Automated prepickup validation run.

    Logic:
    - Cold start creates full 40 µL air gap
    - For each analyte volume:
        - Run all replicates sequentially
        - Wash between every replicate
        - Infuse into consecutive vials on rack1
    """

    print("=== Starting prepickup validation run ===")

    vial_pos = start_vial_pos

    # ---- Cold start only once ----
    print("Cold start: creating full 40 µL air gap")
    rsg.take_air_gap(40)

    for vol in analyte_volumes:
        print(f"\n=== Analyte volume: {vol} µL ===")

        for rep in range(1, n_replicates + 1):
            print(f"  Replicate {rep} → rack1 vial {vial_pos}")

            # 1) Regenerate air gap (skip ONLY for very first run)
            if not (vol == analyte_volumes[0] and rep == 1):
                rsg.take_air_gap(10)

            # 2) Wash (leaves WF + 40 µL air gap)
            rsg.wash_sequence()

            # 3) Prepickup
            rsg.prepickup(volume=prepickup_volume)

            # 4) Small air gap before analyte
            rsg.take_air_gap(5)

            # 5) Pickup analyte
            rsg.pickup_from_vial(
                module_name="rack1",
                vial_pos=1,
                volume=vol,
            )

            # 6) Dispense everything into dilution vial
            dispense_volume = (
                vol                 # analyte
                + 5                 # small air gap
                + prepickup_volume  # prepickup
                + 10                # portion of 40 µL air gap
            )

            rsg.dispense_in_vial(
                module_name="rack1",
                vial_pos=vial_pos,
                volume=dispense_volume,
            )

            vial_pos += 1
            time.sleep(2)

    print("\n=== Prepikcup validation complete ===")


In [ ]:
analyte_volumes = [50, 40, 30, 20, 10]

run_prepickup_validation(
    rsg=rsg,
    analyte_volumes=analyte_volumes,
    start_vial_pos=17,
    n_replicates=3,
    prepickup_volume=20,
)

# Note - the final state of this for-loop based automated run is working fluid + 30uL of the
# 40uL air gap left in the syringe. To get ready for further runs, run the two cells below this one.

In [ ]:
rsg.take_air_gap(10)

In [ ]:
rsg.wash_sequence()

In [19]:
g.go_to_vial("rack1", 1)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130


(np.float64(36.0), np.float64(8.0))

In [13]:
rate = 0.25          # mL/min
volume = 100           # uL
direction = "WDR"     # WDR = withdraw, INF = Infuse

pump.prepare(rate=rate, volume=volume, direction=direction)

Sent: @00*PHN1
Sent: @00*RAT C 0.25 MM
Sent: @00*VOL100
Sent: @00*DIRWDR


In [15]:
pump.start()

Sent: @00*RUN 1


'00W'

In [20]:
g.go_to_vial("rack2", 1)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0


(np.float64(201.0), np.float64(39.0))

In [10]:
g.move_z(130)

'Moved Z to 130. Result: Move Z: 2, Success'

In [79]:
rsg.take_air_gap(10)

Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL10
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP


In [80]:
rsg.wash_sequence()

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL5.0
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
Sent: @00*PHN1
Sent: @00*RAT C 0.5 MM
Sent: @00*VOL100.0
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
Sent: @00*PHN1
Sent: @00*RAT C 0.5 MM
Sent: @00*VOL105.0
Sent: @00*DIRINF
Sent: @00*RUN 1
Sent: @00*STP


In [73]:
rsg.prepickup()

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL10
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP


In [74]:
rsg.take_air_gap(5)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL5
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP


In [19]:
g.go_to_vial("rack1", 1)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130


(np.float64(36.0), np.float64(8.0))

In [20]:
g.move_z(119)

'Moved Z to 119. Result: Move Z: 2, Success'

In [75]:
rsg.pickup_from_vial("rack1", 1, 4)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL4
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP


In [76]:
g.go_to_vial("rack1", 25)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0


(np.float64(52.55), np.float64(159.5295))

In [77]:
g.move_z(119)

'Moved Z to 119. Result: Move Z: 2, Success'

In [78]:
rsg.dispense_in_vial("rack1", 25, 29)

#Note - analyte vol + 5 + 10 + 10 (final 10 is 10uL of the 40uL air gap, leaving 30uL,
# this 10uL is reintroduced before the wash sequence)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
Sent: @00*PHN1
Sent: @00*RAT C 0.5 MM
Sent: @00*VOL29
Sent: @00*DIRINF
Sent: @00*RUN 1
Sent: @00*STP
